# Forward calibration audit (training runs only)

Reconstruct the exact frozen summary and the figure without loading EEG. The separate command `stroke-rehab forward-calibration` regenerates the committed CSVs from the official training recordings. The six runs come from only three participants and test labels were already exposed before this analysis.

In [ ]:
from pathlib import Path
import csv, json
from IPython.display import display, Image
from stroke_rehab.forward_calibration import summarize

here = Path.cwd().resolve()
root = next(path for path in (here, *here.parents)
            if (path / "results/forward_calibration/forward_calibration_summary.csv").is_file())
output = root / "results/forward_calibration"
with (output / "forward_calibration_predictions.csv").open(newline="") as stream:
    predictions = list(csv.DictReader(stream))
with (output / "forward_calibration_summary.csv").open(newline="") as stream:
    summary = list(csv.DictReader(stream))
provenance = json.loads((output / "forward_calibration_provenance.json").read_text())
print("Training files:", sorted(provenance["input_sha256"]))
print("Validation decisions:", len(predictions), "; summary rows:", len(summary))
assert len(predictions) == 3840 and len(summary) == 192
assert all(file.endswith("_training.mat") for file in provenance["input_sha256"])


## Read the fixed evaluation contract

Calibration uses only the first 10, 20, 40, or 60 labels in each run. Every point is scored on the **same** later training trials 61–80. The dashed 50% reference in the figure is for **balanced accuracy**, not raw accuracy (the evaluation tails are not exactly balanced). Filtering and covariance-reference updates are forward only. Feedback timestamps are unavailable even in the early post-cue interval.

In [ ]:
converted = [{**r, "decision_time_s": float(r["decision_time_s"]),
              "calibration_trials": int(r["calibration_trials"]),
              "validation_trial_1based": int(r["validation_trial_1based"]),
              "correct": int(r["correct"]), "true_label": int(r["true_label"])}
             for r in predictions]
computed = summarize(converted)
for row in computed:
    expected = next(s for s in summary if
        (s["patient"], s["session"], s["model"], float(s["decision_time_s"]),
         int(s["calibration_trials"])) ==
        (row["patient"], row["session"], row["model"], row["decision_time_s"],
         row["calibration_trials"]))
    assert int(expected["correct"]) == row["correct"]
    assert abs(float(expected["balanced_accuracy"]) - row["balanced_accuracy"]) < 1e-12
print("All 192 summary rows reproduce from per-trial decisions.")


## Fixed 60-trial calibration example

These are counts over the *same* 120 later training trials across six sessions. They are not 120 independent people. Compare fixed decisions; **do not select the best window on these validation outcomes** and call it a new held-out result.

In [ ]:
for stop in (2.0, 3.5, 4.25, 6.5):
    line = []
    for model in ("filterbank_csp_shrinkage_lda", "filterbank_riemann_recentered"):
        selected = [r for r in summary if float(r["decision_time_s"]) == stop
                    and int(r["calibration_trials"]) == 60 and r["model"] == model]
        line.append(f"{sum(int(r['correct']) for r in selected)}/120"
                    f"; mean BA={sum(float(r['balanced_accuracy']) for r in selected)/6:.3f}")
    print(f"+{stop:g} s: CSP {line[0]} | Riemannian {line[1]}")
display(Image(filename=str(output / "forward_calibration.png"), width=900))


## Interpretation and limitations

Before the +2 s instruction, the predictions provide a negative control. A later rise does not show that the decoder reads only motor intention: stimulation, visual feedback and movement artifacts may contribute. PRE and POST are repeated observations of three people. The models also use different frequency banks, so their comparison is not a pure geometric ablation. No clinical improvement or unbiased new-person generalization follows.